In [2]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd

# List of woods
woods = [
    "Oak", "Maple", "Walnut", "Mahogany", "Cherry", "Teak", "Rosewood", "Ash", "Birch", "Hickory",
    "Pine", "Cedar", "Redwood", "Spruce", "Fir", "Larch",
    "Ebony", "Bamboo", "Wenge", "Zebrawood"
]

# Base Wikipedia URL
base_url = "https://en.wikipedia.org/wiki/"

def get_wikipedia_info(wood):
    url = base_url + wood
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Failed to retrieve data for {wood}")
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Extract title
    title = soup.find('h1', id="firstHeading").text.strip()
    
    # Extract content (first few paragraphs)
    paragraphs = soup.find_all('p')
    content = "\n".join([p.text.strip() for p in paragraphs[:5] if p.text.strip()])
    
    # Extract publication date (not available directly, so using last modified time)
    last_modified = soup.find('li', id="footer-info-lastmod")
    pub_date = last_modified.text if last_modified else "Unknown"
    
    return {"title": title, "content": content, "publication_date": pub_date}

# Crawl Wikipedia for each wood type
data = {}
for wood in woods:
    print(f"Fetching data for {wood}...")
    info = get_wikipedia_info(wood)
    if info:
        data[wood] = info
        # Save each category data into a text file
        with open(f"{wood}.txt", "w", encoding="utf-8") as f:
            f.write(f"Title: {info['title']}\n\n")
            f.write(f"Publication Date: {info['publication_date']}\n\n")
            f.write(f"Content:\n{info['content']}")

# Save results to a JSON file
with open("woods_wikipedia_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

# Save results to a CSV file
df = pd.DataFrame.from_dict(data, orient='index')
df.to_csv("woods_wikipedia_data.csv", encoding="utf-8", index=True)

print("Crawling complete. Data saved to 'woods_wikipedia_data.json' and 'woods_wikipedia_data.csv'")

# Load JSON data and convert to Pandas DataFrame
with open("woods_wikipedia_data.json", "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

df = pd.DataFrame.from_dict(loaded_data, orient='index')
print(df)

Fetching data for Oak...
Fetching data for Maple...
Fetching data for Walnut...
Fetching data for Mahogany...
Fetching data for Cherry...
Fetching data for Teak...
Fetching data for Rosewood...
Fetching data for Ash...
Fetching data for Birch...
Fetching data for Hickory...
Fetching data for Pine...
Fetching data for Cedar...
Fetching data for Redwood...
Fetching data for Spruce...
Fetching data for Fir...
Fetching data for Larch...
Fetching data for Ebony...
Fetching data for Bamboo...
Fetching data for Wenge...
Fetching data for Zebrawood...
Crawling complete. Data saved to 'woods_wikipedia_data.json' and 'woods_wikipedia_data.csv'
                         title  \
Oak                        Oak   
Maple                    Maple   
Walnut                  Walnut   
Mahogany              Mahogany   
Cherry                  Cherry   
Teak                      Teak   
Rosewood              Rosewood   
Ash                        Ash   
Birch                    Birch   
Hickory           

# Preprocessing

In [6]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\priya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [10]:
import pandas as pd
import spacy
import re
import string

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Load dataset
df = pd.read_csv("dataset/woods_wikipedia_data.csv")

def clean_text(text):
    if pd.isna(text):  # Handle missing values
        return ""

    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = text.translate(str.maketrans("", "", string.punctuation))  # Remove punctuation
    
    # Process text with spaCy
    doc = nlp(text)
    
    # Remove stop words & lemmatize
    words = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    
    return " ".join(words)

# Apply cleaning to content column
df["cleaned_content"] = df["content"].astype(str).apply(clean_text)

# Save the cleaned data
df.to_csv("cleaned_woods_wikipedia_data.csv", index=False)

# Display cleaned DataFrame
print(df[["title", "cleaned_content"]].head())


      title                                    cleaned_content
0       Oak  list quercus specie oak hardwood tree shrub ge...
1     Maple  specie group sectionsalphabetical list species...
2    Walnut  walnut edible seed tree genus juglans family j...
3  Mahogany  mahogany straightgraine reddishbrown timber tr...
4    Cherry  cherry fruit plant genus prunus fleshy drupe s...


In [13]:
# take last 4 columns 

df = pd.read_csv("cleaned_woods_wikipedia_data.csv")
df = df.iloc[:, -4:]


df.head()



df.to_csv("cleaned_woods_wikipedia_data.csv", index=False)